In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from tqdm import tqdm
import plotly.express as px
import seaborn as sns
from scipy.spatial import ConvexHull
from tools import T_delta_Sconf, compositionTranform, simplexBasis, unique_elements, getEhull

Load the second inference CSS and select the GNN energy model used for the thermodynamic analysis. The model output is stored as energy per cell and is converted to eV/atom.

In [ ]:
inference_file = 'data/inference2_ccs-gnn_predictions_subset.pkl.gz'
full_inference = pd.read_pickle(inference_file)

models_mapper = {
    'nequip_random_ccs_all': 'nequip_random_ccs',
    'allegro_random_ccs_all': 'allegro_random_ccs',
}
full_inference.rename(mapper=models_mapper, axis=1, inplace=True)

model = 'nequip_random_ccs'
assert model in full_inference.columns
assert (full_inference['natoms'] > 0).all()
assert (full_inference['weight'] > 0).all()
full_inference[model] = full_inference[model] / full_inference['natoms']

print(full_inference.shape)
full_inference.groupby('type').size()

### Configurational free energy

For every composition, calculate
$$F_{conf}(T)=-k_BT\ln\sum_i w_i\exp[-E_i/(k_BT)],$$
where $w_i$ is the number of symmetry-equivalent realizations represented by configuration $i$. The calculation includes configurational contributions only; vibrational, electronic, magnetic, and pressure-volume contributions are not included.

In [ ]:
temperatures = np.linspace(0, 1200, 121)
apply_dft_minimum_correction = False
dft_minimum_file = 'data/inference2_ccs-dft-minima.pkl.gz'
hull_tolerance = 1e-5

if apply_dft_minimum_correction:
    dft_minima = pd.read_pickle(dft_minimum_file)
    assert 'formation_energy_per_atom' in dft_minima.columns
else:
    dft_minima = None

print('List of temperatures used:', temperatures)

When enabled, the optional DFT correction shifts all predicted energies of a composition by the same constant so that the GNN minimum coincides with the lowest energy among the available HT#2 DFT configurations. This preserves the GNN-predicted energy differences within that composition. Generate `data/inference2_ccs-dft-minima.pkl.gz` by running `holdout_test_2.ipynb` through its DFT minima export cell; that step requires the original HT#2 VASP results.

In [ ]:
free_energy_rows = []
formulas = []
formula_groups = full_inference.groupby('formula_str', sort=False)

for formula, subset in tqdm(formula_groups, total=formula_groups.ngroups):
    formulas.append(formula)
    assert subset['natoms'].nunique() == 1
    natoms = int(subset['natoms'].iloc[0])
    energies_per_atom = subset[model].copy()

    if apply_dft_minimum_correction:
        energy_shift = energies_per_atom.min() - dft_minima.loc[formula, 'formation_energy_per_atom']
        energies_per_atom = energies_per_atom - energy_shift

    energies_per_cell = energies_per_atom * natoms
    row = [
        T_delta_Sconf(energies_per_cell, subset['weight'].values, temperature,
                      natoms_for_per_atom=natoms)
        for temperature in temperatures
    ]
    free_energy_rows.append(row)

free_energy_columns = {temperature: f'F_conf_{temperature:g}' for temperature in temperatures}
free_energy_table = pd.DataFrame(
    free_energy_rows,
    index=formulas,
    columns=[free_energy_columns[temperature] for temperature in temperatures],
)

formula_metadata = full_inference.groupby('formula_str').first()
for element in unique_elements:
    free_energy_table[element] = formula_metadata['formula'].apply(lambda formula: formula[element])
free_energy_table['meVac'] = formula_metadata['formula'].apply(lambda formula: formula['U'])
free_energy_table['xVac'] = formula_metadata['formula'].apply(lambda formula: formula['Np'])
free_energy_table['type'] = formula_metadata['type']
free_energy_table['numStrBefMerge'] = full_inference.groupby('formula_str')['weight'].sum()
free_energy_table['numStrAftMerge'] = full_inference.groupby('formula_str').size()
free_energy_table.head()

### Finite-temperature convex hull

Map every composition to the three-dimensional Mo–W–S–Se simplex and append $F_{conf}(T)$ as the fourth coordinate. Rebuild the lower convex hull independently at every temperature. Pure-element vertices remain at zero, so the result is a configurational free-energy hull referenced to the 0 K elemental energies.

In [ ]:
simplex_basis = simplexBasis(len(unique_elements) - 1)
composition_coordinates = pd.DataFrame(
    [compositionTranform(row, simplex_basis) for row in free_energy_table[unique_elements].values],
    columns=[f'SC{i}' for i in range(len(simplex_basis) - 1)],
    index=free_energy_table.index,
)

pure_element_coordinates = pd.DataFrame(
    simplex_basis,
    columns=composition_coordinates.columns,
    index=unique_elements,
)

finite_temperature_hull = pd.concat((pure_element_coordinates, composition_coordinates), axis=0)
for temperature in temperatures:
    column = free_energy_columns[temperature]
    finite_temperature_hull[column] = 0.0
    finite_temperature_hull.loc[free_energy_table.index, column] = free_energy_table[column]

finite_temperature_hull.head()

In [ ]:
for temperature in tqdm(temperatures):
    temperature_tag = f'{temperature:g}'
    free_energy_column = free_energy_columns[temperature]
    points = finite_temperature_hull[
        list(composition_coordinates.columns) + [free_energy_column]
    ].values
    hull = ConvexHull(points)

    hull_reference = np.array([
        getEhull(point, hull) for point in points[:, :len(composition_coordinates.columns)]
    ])
    energies_above_hull = points[:, -1] - hull_reference
    energies_above_hull[np.abs(energies_above_hull) <= hull_tolerance] = 0.0

    finite_temperature_hull[f'F_hull_reference_{temperature_tag}'] = hull_reference
    finite_temperature_hull[f'F_above_hull_{temperature_tag}'] = energies_above_hull
    finite_temperature_hull[f'on_hull_{temperature_tag}'] = (
        np.abs(energies_above_hull) <= hull_tolerance
    )

assert finite_temperature_hull[
    [f'F_above_hull_{temperature:g}' for temperature in temperatures]
].min().min() >= -10 * hull_tolerance
finite_temperature_hull.head()

Plot temperature dependencies of $F_{above hull}$ for complete and reduced configuration sets. Curves from reduced sets contain fewer configurations and therefore provide an approximate configurational contribution.

In [ ]:
vacancy_colors = {'0 : 0': 'k', '1 : 0': 'orange', '0 : 1': 'purple'}
line_styles = {'less60k': '-', 'more60k_highestSG_1': ':'}

plt.figure(figsize=(5, 4))
for set_type in free_energy_table['type'].unique():
    for me_vac, x_vac in [(0, 0), (0, 1), (1, 0)]:
        subset = free_energy_table[
            (free_energy_table['type'] == set_type)
            & (free_energy_table['meVac'] == me_vac)
            & (free_energy_table['xVac'] == x_vac)
        ]
        tag = f'{me_vac} : {x_vac}'
        for formula_id, formula in enumerate(subset.index):
            values = finite_temperature_hull.loc[
                formula,
                [f'F_above_hull_{temperature:g}' for temperature in temperatures],
            ].astype(float).values
            plt.plot(
                temperatures,
                1000 * values,
                line_styles.get(set_type, '--'),
                color=vacancy_colors[tag],
                alpha=0.35,
                label=f'{tag}, {set_type}' if formula_id == 0 else None,
            )

plt.xlabel('T, K')
plt.ylabel('$F_{above hull}$, meV/atom')
plt.grid(alpha=0.3)
plt.legend(title='$V_{Me}:V_X$', labelspacing=0.2)
plt.tight_layout()
plt.show()

Plot the compositions within 120 meV/atom of the finite-temperature hull. Color gives $F_{above hull}$ and marker size emphasizes structures closer to the hull.

In [ ]:
limit_above_hull = 120e-3
for temperature in temperatures[::60]:
    temperature_tag = f'{temperature:g}'
    column = f'F_above_hull_{temperature_tag}'
    simplex_to_plot = finite_temperature_hull.loc[free_energy_table.index, composition_coordinates.columns].copy()
    simplex_to_plot[column] = finite_temperature_hull.loc[free_energy_table.index, column]
    simplex_to_plot = simplex_to_plot[simplex_to_plot[column] < limit_above_hull]
    simplex_to_plot['size'] = limit_above_hull - simplex_to_plot[column] + 1e-3

    fig = px.scatter_3d(
        simplex_to_plot,
        x='SC0', y='SC1', z='SC2',
        color=column, size='size', size_max=10,
        hover_data=[simplex_to_plot.index],
        color_continuous_scale='inferno',
        range_color=(0, limit_above_hull),
        title=f'T = {temperature:g} K',
    )
    fig.update_traces(marker=dict(line=dict(width=0)), selector=dict(mode='markers'))
    fig.show()

### Temperature-dependent stability

Summarize how many compositions belong to the lower hull at each temperature and show the hull-membership map for compositions that appear on the hull at least once.

In [ ]:
on_hull_columns = [f'on_hull_{temperature:g}' for temperature in temperatures]
on_hull_table = finite_temperature_hull.loc[free_energy_table.index, on_hull_columns].copy()
on_hull_table.columns = temperatures

plt.figure(figsize=(5, 3))
plt.plot(temperatures, on_hull_table.sum(axis=0), '.-')
plt.xlabel('T, K')
plt.ylabel('Number of compositions on the hull')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

on_hull_at_least_once = on_hull_table[on_hull_table.any(axis=1)]
plt.figure(figsize=(10, max(3, 0.15 * on_hull_at_least_once.shape[0])))
sns.heatmap(on_hull_at_least_once.astype(int), cmap='Greys', cbar=False)
plt.xlabel('T, K')
plt.ylabel('Composition')
plt.tight_layout()
plt.show()

### Result table

Collect the temperature-dependent quantities in a long-format table and store them for subsequent analysis.

In [ ]:
result_rows = []
for formula in free_energy_table.index:
    for temperature in temperatures:
        temperature_tag = f'{temperature:g}'
        result_rows.append({
            'formula': formula,
            'T': temperature,
            'F_conf': free_energy_table.loc[formula, free_energy_columns[temperature]],
            'F_hull_reference': finite_temperature_hull.loc[formula, f'F_hull_reference_{temperature_tag}'],
            'F_above_hull': finite_temperature_hull.loc[formula, f'F_above_hull_{temperature_tag}'],
            'on_hull': finite_temperature_hull.loc[formula, f'on_hull_{temperature_tag}'],
            'coverage_type': free_energy_table.loc[formula, 'type'],
            'num_configurations': free_energy_table.loc[formula, 'numStrAftMerge'],
            'sum_weights': free_energy_table.loc[formula, 'numStrBefMerge'],
        })

finite_temperature_results = pd.DataFrame(result_rows)
result_file = 'data/inference2_ccs-finite_temperature_hull.pkl.gz'
finite_temperature_results.to_pickle(result_file)
print(finite_temperature_results.shape, result_file)